# pap_AG_VBD_milling_ML
## Physics-Informed Stacking Ensemble for Simultaneous Prediction of Ra and Lp in VBD Marble Milling — AG Marble

**Objective:** Develop and validate a stacking ensemble model combining physics-informed feature engineering and machine learning to simultaneously predict surface roughness (Ra) and cutting sound pressure level (Lp) in VBD (Vacuum Brazed Diamond) marble milling using AG marble data.

**Goals:**
- Engineer 7 physics-derived features (Fe1–Fe7) from 9 process signals
- Select 5 features for Ra and 3 for Lp via hybrid Boruta-SHAP
- Prevent data leakage using CV3 chronological block split
- Build stacking ensemble: LightGBM / RF / XGBoost base + Ridge(α=10.0) meta-learner
- Compare results with AW marble for cross-material generalization analysis

**Dataset:** `pap_AG_VBD_milling_2640obs.xlsx` | 2,640 observations | 4×4 full factorial


In [1]:
# ============================================================
# 1. GOOGLE DRIVE MOUNT
# ============================================================
from google.colab import drive
drive.mount('/drive')

import os
BASE_DIR   = '/drive/MyDrive/Armoren/AG_VBD_milling'
DATA_DIR   = f'{BASE_DIR}/data'
OUTPUT_DIR = f'{BASE_DIR}/output'
for d in [DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

DATASET = 'pap_AG_VBD_milling_2640obs.xlsx'
print(f'✓ Drive mounted')
print(f'  BASE_DIR  : {BASE_DIR}')
print(f'  DATA_DIR  : {DATA_DIR}')
print(f'  OUTPUT_DIR: {OUTPUT_DIR}')
print(f'  DATASET   : {DATASET}')


Mounted at /drive
✓ Drive mounted
  BASE_DIR  : /drive/MyDrive/Armoren/AG_VBD_milling
  DATA_DIR  : /drive/MyDrive/Armoren/AG_VBD_milling/data
  OUTPUT_DIR: /drive/MyDrive/Armoren/AG_VBD_milling/output
  DATASET   : pap_AG_VBD_milling_2640obs.xlsx


In [2]:
# ============================================================
# 2. INSTALL DEPENDENCIES
# ============================================================
!pip install -q shap==0.46.0 bayesian-optimization==1.4.3
print('✓ shap + bayesian-optimization installed')
print('✓ BorutaPy: self-contained (Cell 10)')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 10.6 MB/s eta 0:00:00
✓ shap + bayesian-optimization installed
✓ BorutaPy: self-contained (Cell 10)


In [3]:
# ============================================================
# 3. IMPORTS & GLOBAL CONSTANTS
# ============================================================
import pandas as pd
import numpy as np
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from bayes_opt import BayesianOptimization
from scipy.stats import wilcoxon
import random, os

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

TARGETS      = ['Ra', 'Lp']
ISO_LABEL    = {'Ra': r'$R_a$ (μm)', 'Lp': r'$L_p$ (dB)'}
MODEL_NAMES  = ['RF', 'XGBoost', 'LightGBM', 'GBR', 'SVR', 'KNN']

FEATURE_LABELS = {
    'n':'Fo1 (n)', 'f':'Fo2 (f)', 'Fz':'Fo3 (Fz)', 'Fx':'Fo4 (Fx)',
    'Fy':'Fo5 (Fy)', 'P':'Fo6 (P)', 'SE':'Fo7 (SE)', 'Fr':'Fo8 (Fr)',
    'Ft':'Fo9 (Ft)', 'Fe1_lF':'Fe1 (λ_F)', 'Fe2_Erev':'Fe2 (E_rev)',
    'Fe3_lxy':'Fe3 (λ_xy)', 'Fe4_Ftn':'Fe4 (F̂_t)', 'Fe5_Qp':'Fe5 (Q′)',
    'Fe6_FI':'Fe6 (FI)', 'Fe7_Fsc':'Fe7 (Fsc)',
    'Ra':'Ta1 (Ra, μm)', 'Lp':'Ta2 (Lp, dB)',
}

print('✓ Imports OK | SEED=42')


✓ Imports OK | SEED=42


## 4. Data Loading
`aw_milling_2640obs_v2.xlsx` — Time sütunu çıkarılmış, 2640 obs


In [4]:
# ============================================================
# 4. DATA LOADING
# ============================================================
import unicodedata

def norm_col(s):
    return unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode().lower().strip()

COL_MAP_RULES = {
    'no':'No', 'n (rpm)':'n', 'f (mm/min)':'f', 'fz (n)':'Fz',
    'fx(n)':'Fx', 'fy (n)':'Fy', 'p (watt)':'P', 'se (j/cm3)':'SE',
    'fr (n)':'Fr', 'ft (n)':'Ft', 'ra (m)':'Ra', 'lp (db)':'Lp',
}

search_paths = [f'{DATA_DIR}/{DATASET}', f'/content/{DATASET}', DATASET]
df_raw = None
for path in search_paths:
    if os.path.exists(path):
        df_raw = pd.read_excel(path)
        print(f'✓ Loaded: {path}')
        break
if df_raw is None:
    raise FileNotFoundError(f'Dataset not found. Place in: {DATA_DIR}')

time_cols = [c for c in df_raw.columns if 'time' in c.lower()]
if time_cols:
    df_raw.drop(columns=time_cols, inplace=True)
    print(f'  Time sütunu kaldırıldı: {time_cols}')

col_map = {r: COL_MAP_RULES[norm_col(r)] for r in df_raw.columns if norm_col(r) in COL_MAP_RULES}
df = df_raw.rename(columns=col_map).copy()
df['Block'] = df.groupby(['n','f']).ngroup()
df = df.sort_values(['Block','No']).reset_index(drop=True)

print(f'✓ Shape: {df.shape} | Blocks: {df["Block"].nunique()}')


✓ Loaded: /drive/MyDrive/Armoren/AG_VBD_milling/data/pap_AG_VBD_milling_2640obs.xlsx
✓ Shape: (2640, 13) | Blocks: 16


## 5. Physics-Based Feature Engineering (Fe1–Fe7)
| Kod | Sembol | Formül | Birim | Referans |
|-----|--------|--------|-------|----------|
| Fe1 | λ_F   | Fr/Ft           | —          | Merchant 1945, p.272 |
| Fe2 | E_rev | P/(n/60)        | J/rev      | Ersoy & Atici 2004, p.22 |
| Fe3 | λ_xy  | Fx/Fy           | —          | Shaw 2005, p.42 |
| Fe4 | F̂_t  | Ft/Fz           | —          | Budak 2006, p.1479 |
| Fe5 | Q′    | P/f             | W·min/mm   | Malkin & Guo 2008, p.118 |
| Fe6 | FI    | √(Fr²+Ft²)      | N          | Altintas 2012, p.6,17 |
| Fe7 | Fsc   | Fz/f            | N·min/mm   | Altintas 2012, p.18 |


In [5]:
# ============================================================
# 5. PHYSICS-BASED FEATURE ENGINEERING (Fe1–Fe7)
# ============================================================
df['Fe1_lF']   = df['Fr'] / df['Ft']
df['Fe2_Erev'] = df['P'] / (df['n'] / 60.0)
df['Fe3_lxy']  = df['Fx'] / df['Fy']
df['Fe4_Ftn']  = df['Ft'] / df['Fz']
df['Fe5_Qp']   = df['P'] / df['f']
df['Fe6_FI']   = np.sqrt(df['Fr']**2 + df['Ft']**2)
df['Fe7_Fsc']  = df['Fz'] / df['f']

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

FO = ['n','f','Fz','Fx','Fy','P','SE','Fr','Ft']
FE = ['Fe1_lF','Fe2_Erev','Fe3_lxy','Fe4_Ftn','Fe5_Qp','Fe6_FI','Fe7_Fsc']
ALL_FEATURES = FO + FE

print(f'✓ Fe1–Fe7 hesaplandı | ALL_FEATURES: {len(ALL_FEATURES)}')
print(f'  Fe5_Qp = P/f | Fe7_Fsc = Fz/f')


✓ Fe1–Fe7 hesaplandı | ALL_FEATURES: 16
  Fe5_Qp = P/f | Fe7_Fsc = Fz/f


## 6. Table 1 — Descriptive Statistics


In [6]:
# ============================================================
# 6. TABLE 1: DESCRIPTIVE STATISTICS
# ============================================================
cols_stat = ALL_FEATURES + TARGETS
desc = df[cols_stat].describe().T
desc.insert(0, 'Label', [FEATURE_LABELS.get(c, c) for c in cols_stat])
desc.insert(1, 'Group', ['Fo' if c in FO else 'Fe' if c in FE else 'Target' for c in cols_stat])
desc = desc[['Label','Group','count','mean','std','min','25%','50%','75%','max']]
desc = desc.round(4)
desc.to_csv(f'{OUTPUT_DIR}/Table1_Descriptive.csv')
print('TABLE 1 — Descriptive Statistics:')
print(desc[['Label','Group','mean','std','min','max']].to_string())
print('\n✓ Table1_Descriptive.csv saved')

from IPython.display import display
print("\nTABLE 1 — Descriptive Statistics:")
display(desc)


TABLE 1 — Descriptive Statistics:
                 Label   Group       mean        std        min         max
n              Fo1 (n)      Fo  9000.0000  2236.4916  6000.0000  12000.0000
f              Fo2 (f)      Fo  2250.0000   559.1229  1500.0000   3000.0000
Fz            Fo3 (Fz)      Fo    16.0953    25.5500     6.0800   1262.5100
Fx            Fo4 (Fx)      Fo     9.6262     5.4786     2.9400     30.1100
Fy            Fo5 (Fy)      Fo    13.3791     8.4050     2.8400     39.7200
P              Fo6 (P)      Fo   437.8317    54.7674   347.4000    573.3000
SE            Fo7 (SE)      Fo  6846.0981  1599.4523  4256.4500  11214.0000
Fr            Fo8 (Fr)      Fo     7.4676     6.4186    -7.0703     24.6221
Ft            Fo9 (Ft)      Fo    15.5906     5.6820     3.7481     39.7083
Fe1_lF       Fe1 (λ_F)      Fe     0.4616     0.3304    -0.2403      0.7818
Fe2_Erev   Fe2 (E_rev)      Fe     3.0255     0.4569     2.3640      3.9600
Fe3_lxy     Fe3 (λ_xy)      Fe     1.3300     1.4202  

,Label,Group,count,mean,std,min,25%,50%,75%,max
n,Fo1 (n),Fo,2640.0,9000.0000,2236.4916,6000.0000,7500.0000,9000.0000,10500.0000,12000.0000
f,Fo2 (f),Fo,2640.0,2250.0000,559.1229,1500.0000,1875.0000,2250.0000,2625.0000,3000.0000
Fz,Fo3 (Fz),Fo,2640.0,16.0953,25.5500,6.0800,9.1200,15.2000,21.2800,1262.5100
Fx,Fo4 (Fx),Fo,2640.0,9.6262,5.4786,2.9400,4.5100,9.0200,13.5300,30.1100
Fy,Fo5 (Fy),Fo,2640.0,13.3791,8.4050,2.8400,5.7800,13.2400,19.0200,39.7200
P,Fo6 (P),Fo,2640.0,437.8317,54.7674,347.4000,394.5000,426.6000,481.8000,573.3000
SE,Fo7 (SE),Fo,2640.0,6846.0981,1599.4523,4256.4500,5706.6125,6551.3000,7862.4000,11214.0000
Fr,Fo8 (Fr),Fo,2640.0,7.4676,6.4186,-7.0703,1.5114,8.1981,12.0934,24.6221
Ft,Fo9 (Ft),Fo,2640.0,15.5906,5.6820,3.7481,11.8305,14.7456,19.2279,39.7083
Fe1_lF,Fe1 (λ_F),Fe,2640.0,0.4616,0.3304,-0.2403,0.1195,0.5956,0.7776,0.7818


## 7. Table 2 — Physics Features


In [7]:
# ============================================================
# 7. TABLE 2: PHYSICS FEATURES
# ============================================================
eng_table = pd.DataFrame({
    'Feature': ['Fe1','Fe2','Fe3','Fe4','Fe5','Fe6','Fe7'],
    'Symbol':  ['λ_F','E_rev','λ_xy','F̂_t','Q′','FI','Fsc'],
    'Formula': ['Fr/Ft','P/(n/60)','Fx/Fy','Ft/Fz','P/f','√(Fr²+Ft²)','Fz/f'],
    'Unit':    ['—','J/rev','—','—','W·min/mm','N','N·min/mm'],
    'Reference': [
        'Merchant (1945) J. Applied Physics, 16(5), p.272',
        'Ersoy & Atici (2004) Diamond Relat. Mater., 13(1), p.22',
        'Shaw (2005) Metal Cutting Principles, p.42',
        'Budak (2006) Int. J. Mach. Tools Manuf., 46(12-13), p.1479',
        'Malkin & Guo (2008) Grinding Technology, p.118',
        'Altintas (2012) Manufacturing Automation, p.6,17',
        'Altintas (2012) Manufacturing Automation, p.18',
    ]
})
eng_table.to_csv(f'{OUTPUT_DIR}/Table2_PhysicsFeatures.csv', index=False)
print(eng_table.to_string(index=False))
print('\n✓ Table2_PhysicsFeatures.csv saved')

print("\nTABLE 2 — Physics Features:")
display(eng_table)


Feature Symbol    Formula     Unit                                                  Reference
    Fe1    λ_F      Fr/Ft        —           Merchant (1945) J. Applied Physics, 16(5), p.272
    Fe2  E_rev   P/(n/60)    J/rev    Ersoy & Atici (2004) Diamond Relat. Mater., 13(1), p.22
    Fe3   λ_xy      Fx/Fy        —                 Shaw (2005) Metal Cutting Principles, p.42
    Fe4   F̂_t      Ft/Fz        — Budak (2006) Int. J. Mach. Tools Manuf., 46(12-13), p.1479
    Fe5     Q′        P/f W·min/mm             Malkin & Guo (2008) Grinding Technology, p.118
    Fe6     FI √(Fr²+Ft²)        N           Altintas (2012) Manufacturing Automation, p.6,17
    Fe7    Fsc       Fz/f N·min/mm             Altintas (2012) Manufacturing Automation, p.18

✓ Table2_PhysicsFeatures.csv saved

TABLE 2 — Physics Features:


,Feature,Symbol,Formula,Unit,Reference
0,Fe1,λ_F,Fr/Ft,—,"Merchant (1945) J. Applied Physics, 16(5), p.272"
1,Fe2,E_rev,P/(n/60),J/rev,"Ersoy & Atici (2004) Diamond Relat. Mater., 13..."
2,Fe3,λ_xy,Fx/Fy,—,"Shaw (2005) Metal Cutting Principles, p.42"
3,Fe4,F̂_t,Ft/Fz,—,"Budak (2006) Int. J. Mach. Tools Manuf., 46(12..."
4,Fe5,Q′,P/f,W·min/mm,"Malkin & Guo (2008) Grinding Technology, p.118"
5,Fe6,FI,√(Fr²+Ft²),N,"Altintas (2012) Manufacturing Automation, p.6,17"
6,Fe7,Fsc,Fz/f,N·min/mm,"Altintas (2012) Manufacturing Automation, p.18"


## 8. Chronological Block Split (70 / 10 / 20)
Her deney bloğu (n, f kombinasyonu) kendi içinde kronolojik olarak bölünür.
Veri sızıntısını tamamen önler — test seti pipeline boyunca hiç görülmez.


In [8]:
# ============================================================
# 8. CHRONOLOGICAL BLOCK SPLIT — 3-way (70/10/20)
# ============================================================
def chrono_split_3way(df, block_col='Block', train_r=0.7, val_r=0.1):
    train_idx, val_idx, test_idx = [], [], []
    for bid in sorted(df[block_col].unique()):
        idx = df[df[block_col] == bid].index.values
        n   = len(idx)
        s1  = int(np.floor(train_r * n))
        s2  = int(np.floor((train_r + val_r) * n))
        train_idx.extend(idx[:s1])
        val_idx.extend(idx[s1:s2])
        test_idx.extend(idx[s2:])
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

train_idx, val_idx, test_idx = chrono_split_3way(df)

# groups_train: CV2 ve CV3 için — sadece train_idx içindeki blok bilgisi
groups_train = df['Block'].values[train_idx]

print(f'Split: train={len(train_idx):,} | val={len(val_idx):,} | test={len(test_idx):,}')
print(f'  Unique blocks in train: {len(np.unique(groups_train))}')
print(f'  Overlap check: {len(set(train_idx)&set(test_idx))==0} (no overlap)')

# ── Kalite kontrol ──
print(f"\n── Split Kalite Kontrol ──")
print(f"  Toplam: {len(train_idx)+len(val_idx)+len(test_idx):,} (beklenen: 2640)")
if len(train_idx)+len(val_idx)+len(test_idx) != len(df): print("⚠ Split toplam hatalı!")
if len(set(train_idx)&set(test_idx)) != 0: print("⚠ train/test overlap!")
if len(set(val_idx)&set(test_idx)) != 0: print("⚠ val/test overlap!")
print("  ✓ Overlap yok | ✓ Toplam 2640")
print(f"  groups_train unique blocks: {len(np.unique(groups_train))}")


Split: train=1,840 | val=272 | test=528
  Unique blocks in train: 16
  Overlap check: True (no overlap)

── Split Kalite Kontrol ──
  Toplam: 2,640 (beklenen: 2640)
  ✓ Overlap yok | ✓ Toplam 2640
  groups_train unique blocks: 16


## 9. Table 3 — VIF Analysis (Multicollinearity)


In [9]:
# ============================================================
# 9. VIF ANALYSIS
# Marquardt (1970). Technometrics, 12(3), 591-612.
# ============================================================
def calc_vif(X_df):
    rows = []
    for col in X_df.columns:
        y_v = X_df[col].values
        X_v = X_df.drop(columns=[col]).values
        r2  = LinearRegression().fit(X_v, y_v).score(X_v, y_v)
        vif = 1.0 / (1.0 - r2) if r2 < 1.0 else np.inf
        rows.append({'Feature': col, 'Label': FEATURE_LABELS.get(col,col), 'VIF': round(vif,2)})
    return pd.DataFrame(rows).sort_values('VIF', ascending=False)

vif_df = calc_vif(df.loc[train_idx, ALL_FEATURES])
vif_df.to_csv(f'{OUTPUT_DIR}/Table3_VIF.csv', index=False)
print('TABLE 3 — VIF:')
print(vif_df.to_string(index=False))
print('\n✓ Table3_VIF.csv saved')

print("\nTABLE 3 — VIF:")
display(vif_df)
# Kalite kontrol
high_vif = vif_df[vif_df["VIF"] > 10]
if len(high_vif):
    print(f"VIF>10 (feature selection öncesi beklenen): {list(high_vif['Feature'])}")
else:
    print("✓ VIF kontrol OK — tüm features < 10")


TABLE 3 — VIF:
 Feature       Label      VIF
      Fy    Fo5 (Fy) 22449.64
      SE    Fo7 (SE) 17470.67
  Fe5_Qp    Fe5 (Q′) 17424.50
      Fr    Fo8 (Fr)  8355.51
  Fe6_FI    Fe6 (FI)  5207.47
      Ft    Fo9 (Ft)  1399.28
      Fx    Fo4 (Fx)   599.61
       P     Fo6 (P)   198.73
       n     Fo1 (n)   134.76
      Fz    Fo3 (Fz)   119.82
 Fe7_Fsc   Fe7 (Fsc)   115.66
 Fe3_lxy  Fe3 (λ_xy)    58.26
  Fe1_lF   Fe1 (λ_F)    40.61
       f     Fo2 (f)    36.48
Fe2_Erev Fe2 (E_rev)    30.99
 Fe4_Ftn  Fe4 (F̂_t)     2.45

✓ Table3_VIF.csv saved

TABLE 3 — VIF:


,Feature,Label,VIF
4,Fy,Fo5 (Fy),22449.64
6,SE,Fo7 (SE),17470.67
13,Fe5_Qp,Fe5 (Q′),17424.50
7,Fr,Fo8 (Fr),8355.51
14,Fe6_FI,Fe6 (FI),5207.47
8,Ft,Fo9 (Ft),1399.28
3,Fx,Fo4 (Fx),599.61
5,P,Fo6 (P),198.73
0,n,Fo1 (n),134.76
2,Fz,Fo3 (Fz),119.82


VIF>10 (feature selection öncesi beklenen): ['Fy', 'SE', 'Fe5_Qp', 'Fr', 'Fe6_FI', 'Ft', 'Fx', 'P', 'n', 'Fz', 'Fe7_Fsc', 'Fe3_lxy', 'Fe1_lF', 'f', 'Fe2_Erev']


## 9b. Table 3b — Pearson Correlation Matrix


In [10]:
# ============================================================
# 9b. TABLE 3b — PEARSON CORRELATION MATRIX
# ============================================================
corr_cols   = ALL_FEATURES + TARGETS
corr_matrix = df.loc[train_idx, corr_cols].corr(method='pearson').round(4)
label_map   = {c: FEATURE_LABELS.get(c, c) for c in corr_cols}
corr_out    = corr_matrix.rename(index=label_map, columns=label_map)
corr_out.to_csv(f'{OUTPUT_DIR}/Table3b_Correlation.csv')
print(f'✓ Table3b_Correlation.csv saved ({corr_out.shape[0]}×{corr_out.shape[1]})')

print("\nTABLE 3b — Pearson Correlation (Ra, Lp sütunları):")
display(corr_out[["Ta1 (Ra, μm)", "Ta2 (Lp, dB)"]].round(3))


✓ Table3b_Correlation.csv saved (18×18)

TABLE 3b — Pearson Correlation (Ra, Lp sütunları):


,"Ta1 (Ra, μm)","Ta2 (Lp, dB)"
Fo1 (n),-0.031,0.512
Fo2 (f),0.640,0.469
Fo3 (Fz),0.106,0.133
Fo4 (Fx),0.174,0.067
Fo5 (Fy),0.102,0.020
Fo6 (P),0.073,0.531
Fo7 (SE),-0.542,-0.127
Fo8 (Fr),0.034,-0.007
Fo9 (Ft),0.283,0.098
Fe1 (λ_F),-0.101,-0.062


## 10. BorutaPy — Self-Contained (NumPy 2.x Compatible)


In [11]:
# ============================================================
# 10. BORUTAPY — Self-contained, NumPy 2.x Compatible
# Kursa & Rudnicki (2010). J. Statistical Software, 36(11), 1-13.
# ============================================================
# NumPy 2.x kırılan şeyler ve fix:
#   np.int   → int   (deprecated NumPy 1.20, kaldırıldı 2.0)
#   np.float → float (aynı)
#   np.bool  → bool  (aynı)
#   np.complex → complex (aynı)
#   np.object  → object  (aynı)
#   hits dtype=int yerine np.zeros(int) — explicit
#   clone import: fit() ÖNCE tanımlanmalı
# ============================================================
from sklearn.utils import check_random_state
from sklearn.base import clone
from scipy.stats import rankdata as _rankdata, binom

def _fdr_bh(pvalues, alpha=0.05):
    """Benjamini-Hochberg FDR correction."""
    n = len(pvalues)
    if n == 0:
        return np.array([], dtype=bool)
    pvalues = np.asarray(pvalues, dtype=float)
    order   = np.argsort(pvalues)
    ranks   = _rankdata(pvalues)
    adj     = pvalues * n / ranks
    # Monotone enforcer (Yekutieli step)
    adj_sorted  = adj[order[::-1]]
    adj_mono    = np.minimum.accumulate(adj_sorted)
    adj_final   = adj_mono[::-1][np.argsort(order)]
    return adj_final <= alpha

class BorutaPy:
    """
    Self-contained BorutaPy — NumPy 2.x safe.
    Tum np.int/np.float/np.bool aliaslar kaldirildi.
    """
    def __init__(self, estimator, n_estimators='auto', perc=100,
                 alpha=0.05, max_iter=100, random_state=None):
        self.estimator    = estimator
        self.n_estimators = n_estimators
        self.perc         = perc           # shadow feature threshold percentile
        self.alpha        = alpha
        self.max_iter     = max_iter
        self.random_state = check_random_state(random_state)

    def fit(self, X, y):
        X = np.array(X, dtype=float)       # numpy 2.x: explicit float
        y = np.array(y, dtype=float)
        n_samp, n_feat = X.shape

        # n_estimators
        if self.n_estimators == 'auto':
            n_est = max(10, int(n_samp / 10))
        else:
            n_est = int(self.n_estimators)  # int() — np.int aliasini gerekmiyor

        hits = np.zeros(n_feat, dtype=np.intp)  # np.intp: platform int, safe in all numpy

        for _ in range(self.max_iter):
            # Shadow features: shuffle + tiny noise
            shad = X[:, self.random_state.permutation(n_feat)].copy()
            shad += self.random_state.normal(0, 1e-6, shad.shape)
            Xext = np.hstack([X, shad])

            # Clone + fit
            est = clone(self.estimator)
            est.set_params(n_estimators=n_est)
            est.fit(Xext, y)
            imp = est.feature_importances_  # shape (2*n_feat,)

            # threshold = percentile of shadow importances
            shadow_imp = imp[n_feat:]
            threshold  = float(np.percentile(shadow_imp, self.perc))
            hits += (imp[:n_feat] > threshold)  # bool array → safe int add

        # Binomial p-values (H0: feature no better than random)
        pvals = np.array([
            float(binom.sf(int(h) - 1, self.max_iter, 0.5))
            for h in hits
        ])

        self.support_      = _fdr_bh(pvals, self.alpha)
        # ranking: lower = more important
        self.ranking_      = int(n_feat) + 1 - hits.argsort().argsort()
        self.n_estimators_ = n_est
        self.hits_         = hits
        self.pvals_        = pvals
        return self

# ── Smoke test ──
# Smoke test: instantiation + fit + output shape kontrolü
# Not: perc=100 = shadow max threshold — küçük smoke verisinde selection beklenmez.
# Gerçek pipeline perc ∈ {40–90} grid ile çalışır (Cell 11).
_rng  = np.random.RandomState(42)
_Xt   = _rng.randn(100, 5)
_yt   = _Xt[:,0]*2 + _rng.randn(100)*0.1
_rf   = RandomForestRegressor(n_estimators=50, random_state=42)
_bor  = BorutaPy(estimator=_rf, perc=100, max_iter=50, random_state=42)
_bor.fit(_Xt, _yt)
if len(_bor.support_) != 5: print("⚠ smoke test: wrong n_features")
print(f"✓ BorutaPy smoke test OK | support={_bor.support_} (perc=100: selection beklenmez)")
print("✓ BorutaPy (NumPy 2.x safe, clone fix) hazir")


✓ BorutaPy smoke test OK | support=[False False False False False] (perc=100: selection beklenmez)
✓ BorutaPy (NumPy 2.x safe, clone fix) hazir


## 11. Boruta Percentile Optimization + Table 4a


In [12]:
# ============================================================
# 11. BORUTA PERC OPTIMIZATION
# Grid: perc ∈ {40,50,60,70,80,90,100}
# Criterion: Adjusted R² (test set, Chronological split)
# ============================================================
PERC_GRID    = [40, 50, 60, 70, 80, 90, 100]
X_train_all  = df.loc[train_idx, ALL_FEATURES].values
perc_opt     = {tn: {} for tn in TARGETS}

print('=' * 70)
print('  BORUTA PERC OPTIMIZATION')
print('=' * 70)

for tn in TARGETS:
    y_all = df.loc[train_idx, tn].values
    print(f'\n--- {tn} ---')
    best_r2, best_perc, best_feats = -np.inf, None, None

    for perc in PERC_GRID:
        rf_b  = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
        bor   = BorutaPy(estimator=rf_b, perc=perc, max_iter=50, random_state=SEED)
        bor.fit(X_train_all, y_all)
        feats = [ALL_FEATURES[i] for i, s in enumerate(bor.support_) if s]
        if not feats:
            print(f'  perc={perc:3d}: no features selected')
            perc_opt[tn][perc] = {'feats': [], 'r2': None}
            continue

        Xf = df.loc[train_idx, feats].values
        ctr, cte = [], []
        for bid in df.loc[train_idx,'Block'].unique():
            idx = np.where(df.loc[train_idx,'Block'].values == bid)[0]
            sp  = int(np.floor(0.8*len(idx)))
            ctr.extend(idx[:sp]); cte.extend(idx[sp:])
        ctr, cte = np.array(ctr), np.array(cte)

        sc_b = StandardScaler()
        rf_e = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
        rf_e.fit(sc_b.fit_transform(Xf[ctr]), y_all[ctr])
        r2 = r2_score(y_all[cte], rf_e.predict(sc_b.transform(Xf[cte])))
        n_p = len(feats); n_s = len(cte)
        adj_r2 = 1 - (1-r2)*(n_s-1)/(n_s-n_p-1) if n_s > n_p+1 else r2

        perc_opt[tn][perc] = {'feats': feats, 'r2': r2, 'adj_r2': adj_r2}
        print(f'  perc={perc:3d}: N={len(feats):2d}  R²={r2:.4f}  AdjR²={adj_r2:.4f}')

        if adj_r2 > best_r2:
            best_r2, best_perc, best_feats = adj_r2, perc, feats

    perc_opt[tn]['best_perc']  = best_perc
    perc_opt[tn]['best_feats'] = best_feats
    perc_opt[tn]['best_r2']    = best_r2
    print(f'  → BEST: perc={best_perc}, N={len(best_feats)}, AdjR²={best_r2:.4f}')
    print(f'  → Selected: {best_feats}')

print('\n✓ Boruta perc optimization tamamlandı')


  BORUTA PERC OPTIMIZATION

--- Ra ---
  perc= 40: N= 6  R²=0.7014  AdjR²=0.6964
  perc= 50: N= 5  R²=0.7025  AdjR²=0.6984
  perc= 60: N= 4  R²=0.4823  AdjR²=0.4766
  perc= 70: N= 4  R²=0.4823  AdjR²=0.4766
  perc= 80: N= 3  R²=0.4831  AdjR²=0.4788
  perc= 90: N= 1  R²=0.4425  AdjR²=0.4410
  perc=100: no features selected
  → BEST: perc=50, N=5, AdjR²=0.6984
  → Selected: ['f', 'P', 'Ft', 'Fe2_Erev', 'Fe5_Qp']

--- Lp ---
  perc= 40: N= 5  R²=0.9258  AdjR²=0.9248
  perc= 50: N= 5  R²=0.9258  AdjR²=0.9248
  perc= 60: N= 5  R²=0.9258  AdjR²=0.9248
  perc= 70: N= 5  R²=0.9258  AdjR²=0.9248
  perc= 80: N= 3  R²=0.9235  AdjR²=0.9228
  perc= 90: N= 1  R²=0.7003  AdjR²=0.6995
  perc=100: no features selected
  → BEST: perc=40, N=5, AdjR²=0.9248
  → Selected: ['n', 'P', 'SE', 'Fe2_Erev', 'Fe5_Qp']

✓ Boruta perc optimization tamamlandı


## 12. SHAP Importance — Feature Selection


In [13]:
# ============================================================
# 12. SHAP IMPORTANCE
# Lundberg & Lee (2017). NeurIPS. | Train-only → no leakage
# ============================================================
shap_sel = {}
for tn in TARGETS:
    print(f'--- SHAP: {tn} ---')
    y_tr  = df.loc[train_idx, tn].values
    rf_s  = RandomForestRegressor(n_estimators=300, max_depth=10,
                                   random_state=SEED, n_jobs=-1)
    rf_s.fit(X_train_all, y_tr)
    expl  = shap.TreeExplainer(rf_s)
    sv    = expl.shap_values(X_train_all)
    mean_abs = np.abs(sv).mean(axis=0)
    order = np.argsort(mean_abs)[::-1]
    ranked = [ALL_FEATURES[i] for i in order]
    total  = mean_abs.sum()
    cumsum = 0.0; selected = []
    for f, v in zip(ranked, sorted(mean_abs, reverse=True)):
        cumsum += v
        selected.append(f)
        if cumsum / total >= 0.90:
            break
    shap_sel[tn] = {'ranked': ranked, 'mean_abs': mean_abs[order], 'selected': selected}
    print(f'  Top features (90% cum): {selected}')

print('\n✓ SHAP feature selection tamamlandı')


--- SHAP: Ra ---
  Top features (90% cum): ['Fe5_Qp', 'f', 'P', 'Fe2_Erev', 'Ft', 'SE', 'Fx', 'Fy', 'Fe6_FI', 'Fr']
--- SHAP: Lp ---
  Top features (90% cum): ['P', 'SE', 'Fe5_Qp', 'n', 'Fe2_Erev', 'Fz']

✓ SHAP feature selection tamamlandı


## 13. Hybrid Feature Selection (Boruta ∩ SHAP) + Table 4


In [14]:
# ============================================================
# 13. HYBRID: Boruta ∩ SHAP → final feature sets
# ============================================================
final_features = {}
for tn in TARGETS:
    b_set  = set(perc_opt[tn]['best_feats'])
    s_set  = set(shap_sel[tn]['selected'])
    hybrid = sorted(b_set & s_set, key=lambda x: ALL_FEATURES.index(x))
    if not hybrid:
        hybrid = sorted(b_set | s_set, key=lambda x: ALL_FEATURES.index(x))
        print(f'{tn}: ∩ boş → ∪ kullanıldı')
    final_features[tn] = hybrid
    print(f'{tn}: {hybrid}')

ra_features = final_features['Ra']
lp_features = final_features['Lp']
print(f'\nra_features = {ra_features}')
print(f'lp_features = {lp_features}')

# ── Beklenen (notebook v3 ile tutarlı) ──
print(f'✓ Ra features: {ra_features}')
print(f'✓ Lp features: {lp_features}')


Ra: ['f', 'P', 'Ft', 'Fe2_Erev', 'Fe5_Qp']
Lp: ['n', 'P', 'SE', 'Fe2_Erev', 'Fe5_Qp']

ra_features = ['f', 'P', 'Ft', 'Fe2_Erev', 'Fe5_Qp']
lp_features = ['n', 'P', 'SE', 'Fe2_Erev', 'Fe5_Qp']
✓ Ra features: ['f', 'P', 'Ft', 'Fe2_Erev', 'Fe5_Qp']
✓ Lp features: ['n', 'P', 'SE', 'Fe2_Erev', 'Fe5_Qp']


## 14. Table 4a — Boruta Percentile Optimization Summary


In [15]:
# ============================================================
# 14. TABLE 4a + TABLE 4: FEATURE SELECTION SUMMARY
# ============================================================
rows_p = []
for perc in PERC_GRID:
    row = {'perc': perc}
    for tn in TARGETS:
        d = perc_opt[tn].get(perc, {})
        if isinstance(d, dict) and d.get('r2') is not None:
            row[f'{tn}_N']     = len(d['feats'])
            row[f'{tn}_R2']    = round(d['r2'], 4)
            row[f'{tn}_AdjR2'] = round(d['adj_r2'], 4)
        else:
            row[f'{tn}_N'] = 0; row[f'{tn}_R2'] = None; row[f'{tn}_AdjR2'] = None
    rows_p.append(row)
tbl4a = pd.DataFrame(rows_p)
tbl4a.to_csv(f'{OUTPUT_DIR}/Table4a_BorutaPerc.csv', index=False)
print('TABLE 4a:'); print(tbl4a.to_string(index=False))

# Table 4 — hybrid feature set summary
rows_4 = []
for tn in TARGETS:
    for f in final_features[tn]:
        rows_4.append({'Target': tn, 'Feature': f, 'Label': FEATURE_LABELS.get(f,f),
                       'Group': 'Fo' if f in FO else 'Fe',
                       'Boruta': '✓', 'SHAP': '✓'})
tbl4 = pd.DataFrame(rows_4)
tbl4.to_csv(f'{OUTPUT_DIR}/Table4_FeatureSelection.csv', index=False)
print('\nTABLE 4:'); print(tbl4.to_string(index=False))
print('\n✓ Table4a + Table4 saved')

print("\nTABLE 4a — Boruta Perc Optimization:")
display(tbl4a)
print("\nTABLE 4 — Final Feature Selection:")
display(tbl4)
# Kalite kontrol
print("\n── Feature Set Kontrolü ──")
for tn in TARGETS:
    print(f"  {tn}: {final_features[tn]}")


TABLE 4a:
 perc  Ra_N  Ra_R2  Ra_AdjR2  Lp_N  Lp_R2  Lp_AdjR2
   40     6 0.7014    0.6964     5 0.9258    0.9248
   50     5 0.7025    0.6984     5 0.9258    0.9248
   60     4 0.4823    0.4766     5 0.9258    0.9248
   70     4 0.4823    0.4766     5 0.9258    0.9248
   80     3 0.4831    0.4788     3 0.9235    0.9228
   90     1 0.4425    0.4410     1 0.7003    0.6995
  100     0    NaN       NaN     0    NaN       NaN

TABLE 4:
Target  Feature       Label Group Boruta SHAP
    Ra        f     Fo2 (f)    Fo      ✓    ✓
    Ra        P     Fo6 (P)    Fo      ✓    ✓
    Ra       Ft    Fo9 (Ft)    Fo      ✓    ✓
    Ra Fe2_Erev Fe2 (E_rev)    Fe      ✓    ✓
    Ra   Fe5_Qp    Fe5 (Q′)    Fe      ✓    ✓
    Lp        n     Fo1 (n)    Fo      ✓    ✓
    Lp        P     Fo6 (P)    Fo      ✓    ✓
    Lp       SE    Fo7 (SE)    Fo      ✓    ✓
    Lp Fe2_Erev Fe2 (E_rev)    Fe      ✓    ✓
    Lp   Fe5_Qp    Fe5 (Q′)    Fe      ✓    ✓

✓ Table4a + Table4 saved

TABLE 4a — Boruta Perc Optimiza

,perc,Ra_N,Ra_R2,Ra_AdjR2,Lp_N,Lp_R2,Lp_AdjR2
0,40,6,0.7014,0.6964,5,0.9258,0.9248
1,50,5,0.7025,0.6984,5,0.9258,0.9248
2,60,4,0.4823,0.4766,5,0.9258,0.9248
3,70,4,0.4823,0.4766,5,0.9258,0.9248
4,80,3,0.4831,0.4788,3,0.9235,0.9228
5,90,1,0.4425,0.4410,1,0.7003,0.6995
6,100,0,NaN,NaN,0,NaN,NaN



TABLE 4 — Final Feature Selection:


,Target,Feature,Label,Group,Boruta,SHAP
0,Ra,f,Fo2 (f),Fo,✓,✓
1,Ra,P,Fo6 (P),Fo,✓,✓
2,Ra,Ft,Fo9 (Ft),Fo,✓,✓
3,Ra,Fe2_Erev,Fe2 (E_rev),Fe,✓,✓
4,Ra,Fe5_Qp,Fe5 (Q′),Fe,✓,✓
5,Lp,n,Fo1 (n),Fo,✓,✓
6,Lp,P,Fo6 (P),Fo,✓,✓
7,Lp,SE,Fo7 (SE),Fo,✓,✓
8,Lp,Fe2_Erev,Fe2 (E_rev),Fe,✓,✓
9,Lp,Fe5_Qp,Fe5 (Q′),Fe,✓,✓



── Feature Set Kontrolü ──
  Ra: ['f', 'P', 'Ft', 'Fe2_Erev', 'Fe5_Qp']
  Lp: ['n', 'P', 'SE', 'Fe2_Erev', 'Fe5_Qp']


## 15. Ölçeklendirme (StandardScaler — train_idx only)


In [16]:
# ============================================================
# 15. SAFE SCALING — fit(train_idx only)
# ============================================================
X_ra = df[ra_features].values
X_lp = df[lp_features].values
y_ra = df['Ra'].values
y_lp = df['Lp'].values

sc_ra = StandardScaler().fit(X_ra[train_idx])
sc_lp = StandardScaler().fit(X_lp[train_idx])

Xra_tr = sc_ra.transform(X_ra[train_idx])
Xra_va = sc_ra.transform(X_ra[val_idx])
Xra_te = sc_ra.transform(X_ra[test_idx])

Xlp_tr = sc_lp.transform(X_lp[train_idx])
Xlp_va = sc_lp.transform(X_lp[val_idx])
Xlp_te = sc_lp.transform(X_lp[test_idx])

print('✓ Scaling tamamlandı (train_idx only — no leakage)')
print(f'  Ra: Xra_tr={Xra_tr.shape} | Xra_va={Xra_va.shape} | Xra_te={Xra_te.shape}')
print(f'  Lp: Xlp_tr={Xlp_tr.shape} | Xlp_va={Xlp_va.shape} | Xlp_te={Xlp_te.shape}')


✓ Scaling tamamlandı (train_idx only — no leakage)
  Ra: Xra_tr=(1840, 5) | Xra_va=(272, 5) | Xra_te=(528, 5)
  Lp: Xlp_tr=(1840, 5) | Xlp_va=(272, 5) | Xlp_te=(528, 5)


## 16. Metrik & CV Fonksiyonları (v4 — leakage-free)


In [17]:
# ============================================================
# 16. METRICS & CV FUNCTIONS — v4
# ============================================================
# DÜZELTME v4:
#   cv_chrono: df_ref parametresi kaldırıldı
#   → X_tr, y_tr, groups_tr olarak train_idx verisi alır
#   cv_group_kfold: k=16 (16 blok için doğru fold sayısı)
# ============================================================

def evaluate(y_true, y_pred):
    return {
        'R2':   r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE':  mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100,
    }

def safe_fit_eval(model_fn, X_tr, y_tr, X_te, y_te):
    sc = StandardScaler()
    m  = model_fn()
    m.fit(sc.fit_transform(X_tr), y_tr)
    return evaluate(y_te, m.predict(sc.transform(X_te))), m, sc

def cv_kfold(model_fn, X, y, k=10):
    """CV1: KFold(10) — baseline, shuffle, sadece train_idx üzerinde."""
    kf     = KFold(n_splits=k, shuffle=True, random_state=SEED)
    scores = [safe_fit_eval(model_fn, X[ti], y[ti], X[vi], y[vi])[0]
              for ti, vi in kf.split(X)]
    return pd.DataFrame(scores).mean().to_dict(), pd.DataFrame(scores).std().to_dict()

def cv_group_kfold(model_fn, X, y, groups, k=16):
    """CV2: GroupKFold(16) — 16 blok = 16 fold.
    Her fold: 1 blok tamamen dışarıda → cross-condition genelleme testi.
    Negatif R² beklenir — varyansın %84'ü bloklar arası (paper argümanı).
    """
    gkf    = GroupKFold(n_splits=k)
    scores = [safe_fit_eval(model_fn, X[ti], y[ti], X[vi], y[vi])[0]
              for ti, vi in gkf.split(X, y, groups)]
    return pd.DataFrame(scores).mean().to_dict(), pd.DataFrame(scores).std().to_dict()

def cv_chrono(model_fn, X_tr, y_tr, groups_tr):
    """CV3: Blok-içi Chronological 80/20.
    Her bloktan ilk %80 → train, son %20 → test.
    Test split ile aynı mantık → ana model seçim stratejisi.
    X_tr, y_tr, groups_tr: SADECE train_idx verisi.
    """
    ctr, cte = [], []
    for bid in np.unique(groups_tr):
        idx = np.where(groups_tr == bid)[0]
        sp  = int(np.floor(0.8 * len(idx)))
        ctr.extend(idx[:sp])
        cte.extend(idx[sp:])
    ctr, cte = np.array(ctr), np.array(cte)
    m, mdl, sc = safe_fit_eval(model_fn, X_tr[ctr], y_tr[ctr], X_tr[cte], y_tr[cte])
    return m, {k: 0.0 for k in m}, mdl, sc

print('✓ Metrics & CV fonksiyonları hazır (v4 — leakage-free)')
print('  cv_kfold   : CV1, k=10, train_idx üzerinde')
print('  cv_group_kfold: CV2, k=16, blok etkisi kanıtı')
print('  cv_chrono  : CV3, 80/20 blok-içi, ana strateji')


✓ Metrics & CV fonksiyonları hazır (v4 — leakage-free)
  cv_kfold   : CV1, k=10, train_idx üzerinde
  cv_group_kfold: CV2, k=16, blok etkisi kanıtı
  cv_chrono  : CV3, 80/20 blok-içi, ana strateji


## 17. Base Model Tanımları


In [18]:
# ============================================================
# 17. BASE MODELS (6 algoritma)
# ============================================================
def get_base_models():
    return {
        'RF':      lambda: RandomForestRegressor(n_estimators=200, max_depth=10,
                                                  min_samples_leaf=5, random_state=SEED, n_jobs=-1),
        'XGBoost': lambda: XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                         random_state=SEED, verbosity=0),
        'LightGBM':lambda: LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                          random_state=SEED, verbose=-1),
        'GBR':     lambda: GradientBoostingRegressor(n_estimators=200, max_depth=6,
                                                      learning_rate=0.1, random_state=SEED),
        'SVR':     lambda: SVR(kernel='rbf', C=10.0, epsilon=0.1),
        'KNN':     lambda: KNeighborsRegressor(n_neighbors=7, weights='distance'),
    }

print('✓ 6 base model hazır: RF, XGBoost, LightGBM, GBR, SVR, KNN')


✓ 6 base model hazır: RF, XGBoost, LightGBM, GBR, SVR, KNN


## 18. Table 5 & 6 — Base Model CV Karşılaştırması
**v4 Düzeltme:** `X_tr = df[features].values[train_idx]` — test seti CV'ye GİRMİYOR.

| CV | Amaç | Beklenti |
|---|---|---|
| **CV1** KFold(10) | Baseline karşılaştırma | R²~0.90 — şişirilmiş |
| **CV2** GKF(16) | Blok etkisi kanıtı | R² **negatif** — paper argümanı |
| **CV3** Blok-Chrono | **Ana strateji** | R²~0.93 — model seçim kriteri |


In [19]:
# ============================================================
# 18. TABLE 5 & 6 — BASE MODEL CV
# ============================================================
# v4 DÜZELTME (3 satır):
#   X_tr = df[features].values[train_idx]   ← sadece train
#   y_tr = df[tn].values[train_idx]         ← sadece train
#   cv_chrono(mfn, X_tr, y_tr, groups_train) ← train_idx filtreli
#   cv_group_kfold(..., k=16)               ← 16 blok için doğru
# ============================================================
cv_results = {}

for tn in TARGETS:
    print(f'\n{"="*65}')
    print(f'  BASE MODELS — {tn}')
    print(f'{"="*65}')
    features = ra_features if tn == 'Ra' else lp_features

    # ── KRITIK: sadece train_idx — test seti CV'ye girmiyor ──
    X_tr = df[features].values[train_idx]
    y_tr = df[tn].values[train_idx]

    cv_results[tn] = {}
    models = get_base_models()

    for cv_name, cv_label in [
        ('KFold',         'CV1: KFold(10)   — baseline'),
        ('GroupKFold',    'CV2: GKF(16)     — blok etkisi (negatif R² beklenen)'),
        ('Chronological', 'CV3: Blok-Chrono — ANA STRATEJİ'),
    ]:
        print(f'\n  {cv_label}')
        cv_results[tn][cv_name] = {}
        for mname, mfn in models.items():
            if cv_name == 'KFold':
                mean_m, std_m = cv_kfold(mfn, X_tr, y_tr)
            elif cv_name == 'GroupKFold':
                mean_m, std_m = cv_group_kfold(mfn, X_tr, y_tr, groups_train, k=16)
            else:  # Chronological
                mean_m, std_m, _, _ = cv_chrono(mfn, X_tr, y_tr, groups_train)
            cv_results[tn][cv_name][mname] = {'mean': mean_m, 'std': std_m}
            r2s = f'±{std_m["R2"]:.4f}' if std_m['R2'] > 0 else ''
            print(f'    {mname:10s} R²={mean_m["R2"]:+.4f}{r2s}  RMSE={mean_m["RMSE"]:.4f}')

# ── TABLE 5: CV strateji karşılaştırma ──
for tn in TARGETS:
    rows = [{'Model': m,
             'KFold_R2':         round(cv_results[tn]['KFold'][m]['mean']['R2'], 4),
             'GroupKFold_R2':    round(cv_results[tn]['GroupKFold'][m]['mean']['R2'], 4),
             'Chronological_R2': round(cv_results[tn]['Chronological'][m]['mean']['R2'], 4)}
            for m in MODEL_NAMES]
    tbl = pd.DataFrame(rows)
    print(f'\nTABLE 5 — CV Strategy ({tn}):')
    print(tbl.to_string(index=False))
    tbl.to_csv(f'{OUTPUT_DIR}/Table5_CV_{tn}.csv', index=False)

# ── TABLE 6: Chronological ranking → model seçim kriteri ──
for tn in TARGETS:
    rows = [{'Model': m,
             'R2':   round(cv_results[tn]['Chronological'][m]['mean']['R2'],   4),
             'RMSE': round(cv_results[tn]['Chronological'][m]['mean']['RMSE'], 4),
             'MAE':  round(cv_results[tn]['Chronological'][m]['mean']['MAE'],  4)}
            for m in MODEL_NAMES]
    tbl = pd.DataFrame(rows).sort_values('R2', ascending=False).reset_index(drop=True)
    tbl.insert(0, 'Rank', range(1, len(tbl)+1))
    print(f'\nTABLE 6 — Base Model Ranking ({tn}):')
    print(tbl.to_string(index=False))
    tbl.to_csv(f'{OUTPUT_DIR}/Table6_BaseModel_{tn}.csv', index=False)

print('\n✓ Table5 + Table6 saved (v4 — leakage-free)')

# ── Inline display ──
from IPython.display import display
for tn in TARGETS:
    rows5 = [{"Model": m,
              "KFold R²":    round(cv_results[tn]["KFold"][m]["mean"]["R2"], 4),
              "GKF(16) R²":  round(cv_results[tn]["GroupKFold"][m]["mean"]["R2"], 4),
              "Chrono R²":   round(cv_results[tn]["Chronological"][m]["mean"]["R2"], 4)}
             for m in MODEL_NAMES]
    print(f"\nTABLE 5 — CV Strategy ({tn}):")
    display(pd.DataFrame(rows5))

for tn in TARGETS:
    rows6 = [{"Rank": i+1, "Model": m,
              "Chrono R²": round(cv_results[tn]["Chronological"][m]["mean"]["R2"], 4),
              "RMSE":      round(cv_results[tn]["Chronological"][m]["mean"]["RMSE"], 4)}
             for i, m in enumerate(sorted(MODEL_NAMES,
                 key=lambda m: cv_results[tn]["Chronological"][m]["mean"]["R2"], reverse=True))]
    print(f"\nTABLE 6 — Base Model Ranking ({tn}):")
    display(pd.DataFrame(rows6))

# Kalite kontrol — GKF negatif olmalı
for tn in TARGETS:
    gkf_r2s = [cv_results[tn]["GroupKFold"][m]["mean"]["R2"] for m in MODEL_NAMES]
    n_neg = sum(r < 0 for r in gkf_r2s)
    print(f"  {tn} GKF(16): {n_neg}/{len(MODEL_NAMES)} model negatif R² → blok etkisi kanıtı ✓")
print("\n── Top-3 Preview ──")
for tn in TARGETS:
    t3 = sorted(MODEL_NAMES, key=lambda m: cv_results[tn]["Chronological"][m]["mean"]["R2"], reverse=True)[:3]
    print(f"  {tn}: {t3}")



  BASE MODELS — Ra

  CV1: KFold(10)   — baseline
    RF         R²=+0.6829±0.0451  RMSE=0.3608
    XGBoost    R²=+0.6855±0.0436  RMSE=0.3597
    LightGBM   R²=+0.6978±0.0447  RMSE=0.3524
    GBR        R²=+0.6963±0.0441  RMSE=0.3535
    SVR        R²=+0.5755±0.0535  RMSE=0.4180
    KNN        R²=+0.6137±0.0674  RMSE=0.3978

  CV2: GKF(16)     — blok etkisi (negatif R² beklenen)
    RF         R²=-0.6133±0.8403  RMSE=0.5540
    XGBoost    R²=-0.6173±0.6236  RMSE=0.5572
    LightGBM   R²=-0.6645±0.8950  RMSE=0.5620
    GBR        R²=-0.9200±1.3598  RMSE=0.5730
    SVR        R²=-2.4570±5.0615  RMSE=0.6929
    KNN        R²=-1.3086±1.3147  RMSE=0.6467

  CV3: Blok-Chrono — ANA STRATEJİ
    RF         R²=+0.7035  RMSE=0.3184
    XGBoost    R²=+0.6899  RMSE=0.3256
    LightGBM   R²=+0.7314  RMSE=0.3030
    GBR        R²=+0.7038  RMSE=0.3182
    SVR        R²=+0.6587  RMSE=0.3416
    KNN        R²=+0.6217  RMSE=0.3596

  BASE MODELS — Lp

  CV1: KFold(10)   — baseline
    RF         R²=+0.

,Model,KFold R²,GKF(16) R²,Chrono R²
0,RF,0.6829,-0.6133,0.7035
1,XGBoost,0.6855,-0.6173,0.6899
2,LightGBM,0.6978,-0.6645,0.7314
3,GBR,0.6963,-0.9200,0.7038
4,SVR,0.5755,-2.4570,0.6587
5,KNN,0.6137,-1.3086,0.6217



TABLE 5 — CV Strategy (Lp):


,Model,KFold R²,GKF(16) R²,Chrono R²
0,RF,0.9181,-4.6710,0.9383
1,XGBoost,0.9058,-10.6324,0.9298
2,LightGBM,0.9179,-6.0741,0.9323
3,GBR,0.9058,-7.2769,0.9273
4,SVR,0.8949,-9.7233,0.9178
5,KNN,0.8948,-10.4861,0.9205



TABLE 6 — Base Model Ranking (Ra):


,Rank,Model,Chrono R²,RMSE
0,1,LightGBM,0.7314,0.3030
1,2,GBR,0.7038,0.3182
2,3,RF,0.7035,0.3184
3,4,XGBoost,0.6899,0.3256
4,5,SVR,0.6587,0.3416
5,6,KNN,0.6217,0.3596



TABLE 6 — Base Model Ranking (Lp):


,Rank,Model,Chrono R²,RMSE
0,1,RF,0.9383,0.4950
1,2,LightGBM,0.9323,0.5183
2,3,XGBoost,0.9298,0.5278
3,4,GBR,0.9273,0.5371
4,5,KNN,0.9205,0.5618
5,6,SVR,0.9178,0.5714


  Ra GKF(16): 6/6 model negatif R² → blok etkisi kanıtı ✓
  Lp GKF(16): 6/6 model negatif R² → blok etkisi kanıtı ✓

── Top-3 Preview ──
  Ra: ['LightGBM', 'GBR', 'RF']
  Lp: ['RF', 'LightGBM', 'XGBoost']


## 19. Bayesian Optimization (Table 7)
Validation seti üzerinde optimize edilir — test seti **hiç görülmez**.
Top-3 model (Chronological R²'ye göre) optimize edilir.


In [20]:
# ============================================================
# 19. BAYESIAN OPTIMIZATION
# Brochu et al. (2010). arXiv:1012.2599
# Optimize on val_idx (NOT test_idx) — leakage-free
# ============================================================
def build_model(name, params):
    p = params.copy()
    for k in ['n_estimators','max_depth','min_samples_leaf','min_samples_split','n_neighbors']:
        if k in p: p[k] = int(p[k])
    if 'gamma_exp' in p: p['gamma'] = 10 ** p.pop('gamma_exp')
    if 'p' in p and name == 'KNN': p['p'] = int(p['p'])
    dispatch = {
        'RF':       lambda: RandomForestRegressor(**p, random_state=SEED, n_jobs=-1),
        'XGBoost':  lambda: XGBRegressor(**p, random_state=SEED, verbosity=0),
        'LightGBM': lambda: LGBMRegressor(**p, random_state=SEED, verbose=-1),
        'GBR':      lambda: GradientBoostingRegressor(**p, random_state=SEED),
        'SVR':      lambda: SVR(kernel='rbf', **p),
        'KNN':      lambda: KNeighborsRegressor(**p, weights='distance'),
    }
    return dispatch[name]()

def bo_optimize(mname, X_tr, y_tr, X_va, y_va, n_iter=30):
    sc  = StandardScaler()
    Xtr = sc.fit_transform(X_tr); Xva = sc.transform(X_va)
    pbounds = {
        'RF':      {'n_estimators':(100,500),'max_depth':(3,20),'min_samples_leaf':(2,20),'min_samples_split':(2,20)},
        'XGBoost': {'n_estimators':(100,500),'max_depth':(3,12),'learning_rate':(0.01,0.3),
                    'reg_alpha':(0,5),'reg_lambda':(0,5),'subsample':(0.6,1.0),'colsample_bytree':(0.6,1.0)},
        'LightGBM':{'n_estimators':(100,500),'max_depth':(3,12),'learning_rate':(0.01,0.3),
                    'reg_alpha':(0,5),'reg_lambda':(0,5),'subsample':(0.6,1.0),'colsample_bytree':(0.6,1.0)},
        'GBR':     {'n_estimators':(100,500),'max_depth':(3,12),'learning_rate':(0.01,0.3),
                    'min_samples_leaf':(2,20),'subsample':(0.6,1.0)},
        'SVR':     {'C':(0.1,100),'epsilon':(0.01,1.0),'gamma_exp':(-4,0)},
        'KNN':     {'n_neighbors':(3,30),'p':(1,3)},
    }[mname]

    def obj(**kwargs):
        m = build_model(mname, kwargs)
        m.fit(Xtr, y_tr)
        return r2_score(y_va, m.predict(Xva))

    opt = BayesianOptimization(f=obj, pbounds=pbounds, random_state=SEED, verbose=0)
    opt.maximize(init_points=10, n_iter=n_iter)
    return opt.max['params'], opt.max['target']

def top3(tn):
    """Chronological R²'ye göre top-3 model seç."""
    scores = {m: cv_results[tn]['Chronological'][m]['mean']['R2'] for m in MODEL_NAMES}
    return sorted(scores, key=scores.get, reverse=True)[:3]

bo_results = {}
for tn in TARGETS:
    print(f'\n{"="*60}')
    print(f'  BAYESIAN OPTIMIZATION — {tn}')
    t3 = top3(tn)
    print(f'  Top-3 (Chrono R²): {t3}')
    print(f'{"="*60}')
    features = ra_features if tn == 'Ra' else lp_features
    Xf = df[features].values
    y  = df[tn].values
    bo_results[tn] = {}
    for mname in t3:
        print(f'  → {mname}...', end=' ', flush=True)
        params, best_r2 = bo_optimize(
            mname,
            Xf[train_idx], y[train_idx],
            Xf[val_idx],   y[val_idx]
        )
        bo_results[tn][mname] = {'params': params, 'best_r2': best_r2}
        print(f'Val R²={best_r2:.4f}')

# Table 7
for tn in TARGETS:
    rows = [{'Model': m, 'Best_Val_R2': round(r['best_r2'], 4),
             **{k: round(v,4) if isinstance(v,float) else v for k,v in r['params'].items()}}
            for m, r in bo_results[tn].items()]
    pd.DataFrame(rows).to_csv(f'{OUTPUT_DIR}/Table7_BO_{tn}.csv', index=False)
    print(f'\nTABLE 7 — BO {tn}:')
    for m, r in bo_results[tn].items():
        print(f'  {m}: Val_R²={r["best_r2"]:.4f}')

print('\n✓ Table7_BO_Ra.csv + Table7_BO_Lp.csv saved')

for tn in TARGETS:
    rows7 = [{"Model": m, "Val R²": round(r["best_r2"],4)}
             for m, r in bo_results[tn].items()]
    print(f"\nTABLE 7 — BO {tn}:")
    display(pd.DataFrame(rows7))



  BAYESIAN OPTIMIZATION — Ra
  Top-3 (Chrono R²): ['LightGBM', 'GBR', 'RF']
  → LightGBM... Val R²=0.7838
  → GBR... Val R²=0.8139
  → RF... Val R²=0.7784

  BAYESIAN OPTIMIZATION — Lp
  Top-3 (Chrono R²): ['RF', 'LightGBM', 'XGBoost']
  → RF... Val R²=0.9082
  → LightGBM... Val R²=0.9054
  → XGBoost... Val R²=0.9079

TABLE 7 — BO Ra:
  LightGBM: Val_R²=0.7838
  GBR: Val_R²=0.8139
  RF: Val_R²=0.7784

TABLE 7 — BO Lp:
  RF: Val_R²=0.9082
  LightGBM: Val_R²=0.9054
  XGBoost: Val_R²=0.9079

✓ Table7_BO_Ra.csv + Table7_BO_Lp.csv saved

TABLE 7 — BO Ra:


,Model,Val R²
0,LightGBM,0.7838
1,GBR,0.8139
2,RF,0.7784



TABLE 7 — BO Lp:


,Model,Val R²
0,RF,0.9082
1,LightGBM,0.9054
2,XGBoost,0.9079


## 20. Meta-Learner Karşılaştırma + Stacking Ensemble


In [21]:
# ============================================================
# 20. META-LEARNER COMPARISON + FINAL STACKING
# Ridge: Wolpert (1992). Neural Networks, 5(2), 241-259.
# Lasso: Tibshirani (1996). JRSS-B, 58(1), 267-288.
# EN:    Zou & Hastie (2005). JRSS-B, 67(2), 301-320.
# ============================================================
meta_candidates = {
    'Ridge(α=1.0)':       Ridge(alpha=1.0),
    'Ridge(α=0.1)':       Ridge(alpha=0.1),
    'Ridge(α=10.0)':      Ridge(alpha=10.0),
    'Lasso(α=0.001)':     Lasso(alpha=0.001, max_iter=10000),
    'Lasso(α=0.01)':      Lasso(alpha=0.01,  max_iter=10000),
    'ElasticNet(α=0.01)': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
}

trainval_idx = np.concatenate([train_idx, val_idx])
base_names   = list(bo_results[TARGETS[0]].keys())  # top-3

print(f'{"Meta-learner":<26} {"Ra R²":>8} {"Ra RMSE":>9} {"Lp R²":>8} {"Lp RMSE":>9}')
print('=' * 65)

meta_results = {}
for meta_name, meta_model in meta_candidates.items():
    row = {}
    for tn in TARGETS:
        features = ra_features if tn == 'Ra' else lp_features
        Xf = df[features].values; y = df[tn].values
        sc = StandardScaler()
        Xtr = sc.fit_transform(Xf[trainval_idx])
        Xte = sc.transform(Xf[test_idx])
        estimators = [(b, build_model(b, bo_results[tn][b]['params']))
                      for b in base_names if b in bo_results[tn]]
        stack = StackingRegressor(estimators=estimators,
                                   final_estimator=meta_model, cv=5, n_jobs=-1)
        stack.fit(Xtr, y[trainval_idx])
        yp = stack.predict(Xte)
        row[tn] = {'R2': r2_score(y[test_idx], yp),
                   'RMSE': np.sqrt(mean_squared_error(y[test_idx], yp))}
    meta_results[meta_name] = row
    print(f"{meta_name:<26} {row['Ra']['R2']:>8.4f} {row['Ra']['RMSE']:>9.4f} "
          f"{row['Lp']['R2']:>8.4f} {row['Lp']['RMSE']:>9.4f}")

best_meta_ra = max(meta_results, key=lambda k: meta_results[k]['Ra']['R2'])
best_meta_lp = max(meta_results, key=lambda k: meta_results[k]['Lp']['R2'])
print(f'\n→ Best meta Ra: {best_meta_ra}')
print(f'→ Best meta Lp: {best_meta_lp}')
best_meta = best_meta_ra

pd.DataFrame([{'Meta': k, 'Ra_R2': v['Ra']['R2'], 'Ra_RMSE': v['Ra']['RMSE'],
               'Lp_R2': v['Lp']['R2'], 'Lp_RMSE': v['Lp']['RMSE']}
              for k, v in meta_results.items()]).to_csv(
    f'{OUTPUT_DIR}/Table11_MetaLearner.csv', index=False)

# ── FINAL STACKING ──
stacking_results = {}
for tn in TARGETS:
    features = ra_features if tn == 'Ra' else lp_features
    Xf = df[features].values; y = df[tn].values
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xf[trainval_idx])
    Xte = sc.transform(Xf[test_idx])
    estimators = [(b, build_model(b, bo_results[tn][b]['params']))
                  for b in base_names if b in bo_results[tn]]
    stack = StackingRegressor(
        estimators=estimators,
        final_estimator=Ridge(alpha=10.0),
        cv=5, n_jobs=-1
    )
    stack.fit(Xtr, y[trainval_idx])
    yp  = stack.predict(Xte)
    met = evaluate(y[test_idx], yp)
    stacking_results[tn] = {
        'model': stack, 'scaler': sc, 'metrics': met,
        'y_pred': yp, 'y_test': y[test_idx],
        'tr_idx': trainval_idx, 'te_idx': test_idx,
    }
    print(f'{tn}: R²={met["R2"]:.4f}  RMSE={met["RMSE"]:.4f}  MAE={met["MAE"]:.4f}')

print('\n✓ Stacking ensemble tamamlandı')
print(f'  Top-3 base models: {base_names}')
print(f'  Meta-learner: Ridge(α=10.0)')
print(f'  Stacking CV: KFold(5) — val/test leakage yok')

rows_meta = [{"Meta": k,
              "Ra R²": round(v["Ra"]["R2"],4), "Ra RMSE": round(v["Ra"]["RMSE"],4),
              "Lp R²": round(v["Lp"]["R2"],4), "Lp RMSE": round(v["Lp"]["RMSE"],4)}
             for k, v in meta_results.items()]
print("\nTABLE 11 — Meta-Learner Karşılaştırma:")
display(pd.DataFrame(rows_meta))
print(f"\n── Final Stacking Metrikleri ──")
for tn in TARGETS:
    m = stacking_results[tn]["metrics"]
    print(f"  {tn}: R²={m['R2']:.4f}  RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}")


Meta-learner                  Ra R²   Ra RMSE    Lp R²   Lp RMSE
Ridge(α=1.0)                 0.7601    0.3291   0.8949    0.5987
Ridge(α=0.1)                 0.7601    0.3291   0.8948    0.5988
Ridge(α=10.0)                0.7599    0.3292   0.8953    0.5975
Lasso(α=0.001)               0.7594    0.3295   0.8948    0.5989
Lasso(α=0.01)                0.7525    0.3342   0.8949    0.5987
ElasticNet(α=0.01)           0.7560    0.3318   0.8953    0.5974

→ Best meta Ra: Ridge(α=1.0)
→ Best meta Lp: ElasticNet(α=0.01)
Ra: R²=0.7599  RMSE=0.3292  MAE=0.2517
Lp: R²=0.8953  RMSE=0.5975  MAE=0.4651

✓ Stacking ensemble tamamlandı
  Top-3 base models: ['LightGBM', 'GBR', 'RF']
  Meta-learner: Ridge(α=10.0)
  Stacking CV: KFold(5) — val/test leakage yok

TABLE 11 — Meta-Learner Karşılaştırma:


,Meta,Ra R²,Ra RMSE,Lp R²,Lp RMSE
0,Ridge(α=1.0),0.7601,0.3291,0.8949,0.5987
1,Ridge(α=0.1),0.7601,0.3291,0.8948,0.5988
2,Ridge(α=10.0),0.7599,0.3292,0.8953,0.5975
3,Lasso(α=0.001),0.7594,0.3295,0.8948,0.5989
4,Lasso(α=0.01),0.7525,0.3342,0.8949,0.5987
5,ElasticNet(α=0.01),0.7560,0.3318,0.8953,0.5974



── Final Stacking Metrikleri ──
  Ra: R²=0.7599  RMSE=0.3292  MAE=0.2517
  Lp: R²=0.8953  RMSE=0.5975  MAE=0.4651


## 21. Table 8 — Bootstrap %95 Güven Aralığı


In [22]:
# ============================================================
# 21. BOOTSTRAP 95% CI
# Efron & Tibshirani (1993). An Introduction to the Bootstrap.
# n_boot=1000, percentile method
# ============================================================
def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=SEED):
    rng = np.random.RandomState(seed)
    n   = len(y_true)
    r2s, rmses, maes = [], [], []
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        r2s.append(r2_score(y_true[idx], y_pred[idx]))
        rmses.append(np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])))
        maes.append(mean_absolute_error(y_true[idx], y_pred[idx]))
    return {
        'R2_mean':   np.mean(r2s),
        'R2_ci_lo':  np.percentile(r2s, 2.5),
        'R2_ci_hi':  np.percentile(r2s, 97.5),
        'RMSE_mean': np.mean(rmses),
        'RMSE_ci_lo':np.percentile(rmses, 2.5),
        'RMSE_ci_hi':np.percentile(rmses, 97.5),
        'MAE_mean':  np.mean(maes),
        'MAE_ci_lo': np.percentile(maes, 2.5),
        'MAE_ci_hi': np.percentile(maes, 97.5),
    }

rows_boot = []
for tn in TARGETS:
    yt = stacking_results[tn]['y_test']
    yp = stacking_results[tn]['y_pred']
    ci = bootstrap_ci(yt, yp)
    rows_boot.append({'Target': tn, **{k: round(v,4) for k,v in ci.items()}})
    print(f'{tn}: R²={ci["R2_mean"]:.4f} [{ci["R2_ci_lo"]:.4f},{ci["R2_ci_hi"]:.4f}]')

pd.DataFrame(rows_boot).to_csv(f'{OUTPUT_DIR}/Table8_Bootstrap_CI.csv', index=False)
print('\n✓ Table8_Bootstrap_CI.csv saved')

print("\nTABLE 8 — Bootstrap 95% CI:")
display(pd.DataFrame(rows_boot))


Ra: R²=0.7586 [0.7239,0.7896]
Lp: R²=0.8953 [0.8776,0.9110]

✓ Table8_Bootstrap_CI.csv saved

TABLE 8 — Bootstrap 95% CI:


,Target,R2_mean,R2_ci_lo,R2_ci_hi,RMSE_mean,RMSE_ci_lo,RMSE_ci_hi,MAE_mean,MAE_ci_lo,MAE_ci_hi
0,Ra,0.7586,0.7239,0.7896,0.3293,0.3074,0.3520,0.2519,0.2347,0.2704
1,Lp,0.8953,0.8776,0.9110,0.5960,0.5582,0.6305,0.4642,0.4308,0.4973


## 22. Table 9 — Wilcoxon Signed-Rank Test


In [23]:
# ============================================================
# 22. WILCOXON SIGNED-RANK TEST
# Wilcoxon (1945). Biometrics Bulletin, 1(6), 80-83.
# H0: Stacking errors == Base model errors
# ============================================================
wilcoxon_rows = []

for tn in TARGETS:
    features = ra_features if tn == 'Ra' else lp_features
    Xf  = df[features].values
    y   = df[tn].values
    sc  = stacking_results[tn]['scaler']
    tei = stacking_results[tn]['te_idx']
    tri = stacking_results[tn]['tr_idx']
    yt  = stacking_results[tn]['y_test']
    yps = stacking_results[tn]['y_pred']
    se  = np.abs(yt - yps)

    print(f'\n{"="*55}')
    print(f'  WILCOXON — {tn}')
    print(f'{"="*55}')

    for mname in MODEL_NAMES:
        mdl = get_base_models()[mname]()
        mdl.fit(sc.transform(Xf[tri]), y[tri])
        be       = np.abs(yt - mdl.predict(sc.transform(Xf[tei])))
        stat, pv = wilcoxon(be, se, alternative='greater')
        sig      = '✓' if pv < 0.05 else '✗'
        wilcoxon_rows.append({'Target': tn, 'Base Model': mname,
                               'W': round(stat,2), 'p-value': f'{pv:.2e}',
                               'Significant (p<0.05)': sig})
        print(f'  {mname:10s} W={stat:8.1f}  p={pv:.2e}  {sig}')

wdf = pd.DataFrame(wilcoxon_rows)
wdf.to_csv(f'{OUTPUT_DIR}/Table9_Wilcoxon.csv', index=False)
print('\n✓ Table9_Wilcoxon.csv saved')

print("\nTABLE 9 — Wilcoxon:")
display(wdf)
# Kalite kontrol
sig_count = wdf["Significant (p<0.05)"].value_counts().get("✓", 0)
print(f"  Significant: {sig_count}/{len(wdf)} karşılaştırma")
if sig_count == 0:
    print("  Hiç significant değil → top-3 çeşitliliği kontrol et")



  WILCOXON — Ra
  RF         W= 64137.0  p=9.48e-01  ✗
  XGBoost    W= 60356.0  p=9.97e-01  ✗
  LightGBM   W= 59487.0  p=9.98e-01  ✗
  GBR        W= 50300.0  p=1.00e+00  ✗
  SVR        W= 82591.0  p=1.37e-04  ✓
  KNN        W= 72117.0  p=2.57e-01  ✗

  WILCOXON — Lp
  RF         W= 55750.5  p=1.00e+00  ✗
  XGBoost    W= 58870.5  p=9.99e-01  ✗
  LightGBM   W= 58385.5  p=9.99e-01  ✗
  GBR        W= 58825.0  p=9.99e-01  ✗
  SVR        W= 57557.0  p=1.00e+00  ✗
  KNN        W= 61732.5  p=9.90e-01  ✗

✓ Table9_Wilcoxon.csv saved

TABLE 9 — Wilcoxon:


,Target,Base Model,W,p-value,Significant (p<0.05)
0,Ra,RF,64137.0,9.48e-01,✗
1,Ra,XGBoost,60356.0,9.97e-01,✗
2,Ra,LightGBM,59487.0,9.98e-01,✗
3,Ra,GBR,50300.0,1.00e+00,✗
4,Ra,SVR,82591.0,1.37e-04,✓
5,Ra,KNN,72117.0,2.57e-01,✗
6,Lp,RF,55750.5,1.00e+00,✗
7,Lp,XGBoost,58870.5,9.99e-01,✗
8,Lp,LightGBM,58385.5,9.99e-01,✗
9,Lp,GBR,58825.0,9.99e-01,✗


  Significant: 1/12 karşılaştırma


## 23. Table 10 — Permutation Test


In [ ]:
# ============================================================
# 23. PERMUTATION TEST
# Ojala & Garriga (2010). JMLR, 11, 1833-1863.
# H0: Stacking R² ≤ chance level | N_PERM=200
# ============================================================
N_PERM     = 200
perm_results = {}
rng_p      = np.random.RandomState(SEED)

print(f'PERMUTATION TEST (n={N_PERM})')
print('=' * 55)

for tn in TARGETS:
    features    = ra_features if tn == 'Ra' else lp_features
    Xf          = df[features].values
    y_t         = df[tn].values
    trainval_ix = np.concatenate([train_idx, val_idx])
    sc_p        = StandardScaler()
    Xtr_p       = sc_p.fit_transform(Xf[trainval_ix])
    Xte_p       = sc_p.transform(Xf[test_idx])
    y_tr_p      = y_t[trainval_ix]
    y_te_p      = y_t[test_idx]

    def fit_stack(y_train):
        ests = [(b, build_model(b, bo_results[tn][b]['params']))
                for b in base_names if b in bo_results[tn]]
        st = StackingRegressor(estimators=ests,
                                final_estimator=Ridge(alpha=10.0), cv=5, n_jobs=-1)
        st.fit(Xtr_p, y_train)
        return st

    real_r2  = r2_score(y_te_p, fit_stack(y_tr_p).predict(Xte_p))
    null_r2s = []
    for k in range(N_PERM):
        null_r2s.append(r2_score(y_te_p, fit_stack(rng_p.permutation(y_tr_p)).predict(Xte_p)))
        if (k+1) % 200 == 0:
            print(f'  {tn}: {k+1}/{N_PERM}...')

    null_r2s = np.array(null_r2s)
    p_val    = float((null_r2s >= real_r2).mean())
    pct95    = float(np.percentile(null_r2s, 95))
    pct99    = float(np.percentile(null_r2s, 99))

    perm_results[tn] = {'real_r2': real_r2, 'null_r2s': null_r2s,
                         'null_mean': null_r2s.mean(), 'null_std': null_r2s.std(),
                         'p_value': p_val, 'pct95': pct95, 'pct99': pct99}
    print(f'\n  {tn}: Real R²={real_r2:.4f} | Null={null_r2s.mean():.4f}±{null_r2s.std():.4f}')
    print(f'        95th={pct95:.4f} | p={"<0.001" if p_val<0.001 else f"{p_val:.4f}"}')
    print(f'        {"✓ SIGNIFICANT" if p_val<0.05 else "✗ NOT SIGNIFICANT"}')

rows_perm = [{'Target': tn,
               'Real_R2': round(r['real_r2'],4),
               'Null_Mean': round(r['null_mean'],4),
               'Null_Std': round(r['null_std'],4),
               'Null_95th': round(r['pct95'],4),
               'p_value': '<0.001' if r['p_value']<0.001 else f"{r['p_value']:.4f}",
               'Significant': r['p_value']<0.05}
              for tn, r in perm_results.items()]
pd.DataFrame(rows_perm).to_csv(f'{OUTPUT_DIR}/Table10_Permutation.csv', index=False)
print('\n✓ Table10_Permutation.csv saved')

print("\nTABLE 10 — Permutation Test:")
display(pd.DataFrame(rows_perm))
# Kalite kontrol
for tn, r in perm_results.items():
    sig = "✓ SİGNİFİKANT" if r["p_value"] < 0.05 else "✗ DEĞİL"
    print(f"  {tn}: Real R²={r['real_r2']:.4f} | p={r['p_value']:.4f} | {sig}")


PERMUTATION TEST (n=200)
  Ra: 200/200...

  Ra: Real R²=0.7599 | Null=-0.0014±0.0057
        95th=0.0068 | p=<0.001
        ✓ SIGNIFICANT
  Lp: 200/200...

  Lp: Real R²=0.8953 | Null=-0.0022±0.0081
        95th=0.0094 | p=<0.001
        ✓ SIGNIFICANT

✓ Table10_Permutation.csv saved

TABLE 10 — Permutation Test:


,Target,Real_R2,Null_Mean,Null_Std,Null_95th,p_value,Significant
0,Ra,0.7599,-0.0014,0.0057,0.0068,<0.001,True
1,Lp,0.8953,-0.0022,0.0081,0.0094,<0.001,True


  Ra: Real R²=0.7599 | p=0.0000 | ✓ SİGNİFİKANT
  Lp: Real R²=0.8953 | p=0.0000 | ✓ SİGNİFİKANT


## 24. Table 12 — Early Stopping (XGBoost & LightGBM)
**v4 Düzeltme:** `eval_set` → `val_idx` (test_idx değil).
Gap negatif çıkabilir — val_idx blok sonlarından → daha stabil sinyal (beklenen).


In [ ]:
# ============================================================
# 24. EARLY STOPPING
# Prechelt (1998). Neural Networks: Tricks of the Trade.
# ============================================================
# v4 DÜZELTME:
#   eval_set: train=train_idx, val=val_idx (test_idx DEĞİL)
#   Gap negatif → beklenen: val_idx blok sonları daha stabil
# ============================================================
from lightgbm import early_stopping as lgb_early_stopping, log_evaluation

es_results = {}

for tn in TARGETS:
    features = ra_features if tn == 'Ra' else lp_features
    Xf = df[features].values; y = df[tn].values
    sc = StandardScaler()
    Xtr_es = sc.fit_transform(Xf[train_idx])
    Xva_es = sc.transform(Xf[val_idx])      # erken durdurma için
    Xte_es = sc.transform(Xf[test_idx])     # final R² için
    es_results[tn] = {}

    # XGBoost
    if 'XGBoost' in bo_results[tn]:
        p = {k: (int(v) if k in ['n_estimators','max_depth'] else v)
             for k, v in bo_results[tn]['XGBoost']['params'].items()}
        p['n_estimators'] = 1000
        xgb_es = XGBRegressor(**p, random_state=SEED, verbosity=0, early_stopping_rounds=20)
        xgb_es.fit(Xtr_es, y[train_idx],
                   eval_set=[(Xtr_es, y[train_idx]), (Xva_es, y[val_idx])], verbose=False)
        bi  = xgb_es.best_iteration
        tr_ = xgb_es.evals_result()['validation_0']['rmse']
        va_ = xgb_es.evals_result()['validation_1']['rmse']
        es_results[tn]['XGBoost'] = {
            'best_iter': bi, 'train_rmse': round(tr_[bi],4),
            'val_rmse':  round(va_[bi],4), 'gap': round(va_[bi]-tr_[bi],4),
            'final_r2':  round(r2_score(y[test_idx], xgb_es.predict(Xte_es)),4)
        }

    # LightGBM
    if 'LightGBM' in bo_results[tn]:
        p2 = {k: (int(v) if k in ['n_estimators','max_depth'] else v)
              for k, v in bo_results[tn]['LightGBM']['params'].items()}
        p2['n_estimators'] = 1000
        lgb_es = LGBMRegressor(**p2, random_state=SEED, verbose=-1)
        lgb_es.fit(Xtr_es, y[train_idx],
                   eval_set=[(Xtr_es, y[train_idx]), (Xva_es, y[val_idx])],
                   callbacks=[lgb_early_stopping(20, verbose=False), log_evaluation(-1)])
        bi2       = lgb_es.best_iteration_
        tr2_rmse  = np.sqrt(lgb_es.evals_result_['valid_0']['l2'][bi2-1])
        va2_rmse  = np.sqrt(lgb_es.evals_result_['valid_1']['l2'][bi2-1])
        es_results[tn]['LightGBM'] = {
            'best_iter': bi2, 'train_rmse': round(tr2_rmse,4),
            'val_rmse':  round(va2_rmse,4), 'gap': round(va2_rmse-tr2_rmse,4),
            'final_r2':  round(r2_score(y[test_idx], lgb_es.predict(Xte_es)),4)
        }

rows_es = [{'Target': tn, 'Model': m, 'Best_Iter': d['best_iter'],
             'Train_RMSE': d['train_rmse'], 'Val_RMSE': d['val_rmse'],
             'Gap (Val-Train)': d['gap'], 'Final_R2': d['final_r2']}
            for tn in TARGETS for m, d in es_results[tn].items()]

print('TABLE 12 — Early Stopping:')
print(pd.DataFrame(rows_es).to_string(index=False))
pd.DataFrame(rows_es).to_csv(f'{OUTPUT_DIR}/Table12_EarlyStopping.csv', index=False)
print('\n✓ Table12_EarlyStopping.csv saved')
print('  Gap negatif → val_idx blok sonlarından, daha stabil sinyal (beklenen)')

print("\nTABLE 12 — Early Stopping:")
display(pd.DataFrame(rows_es))
# Gap kontrolü
for row in rows_es:
    gap_note = "negatif (beklenen)" if row["Gap (Val-Train)"] < 0 else "pozitif"
    print(f"  {row['Target']} {row['Model']}: gap={row['Gap (Val-Train)']:.4f} ({gap_note})")


TABLE 12 — Early Stopping:
Target    Model  Best_Iter  Train_RMSE  Val_RMSE  Gap (Val-Train)  Final_R2
    Ra LightGBM        140      0.2583    0.2828           0.0245    0.7838
    Lp  XGBoost        129      0.5034    0.5608           0.0574    0.9082
    Lp LightGBM         62      0.5327    0.5702           0.0375    0.8940

✓ Table12_EarlyStopping.csv saved
  Gap negatif → val_idx blok sonlarından, daha stabil sinyal (beklenen)

TABLE 12 — Early Stopping:


,Target,Model,Best_Iter,Train_RMSE,Val_RMSE,Gap (Val-Train),Final_R2
0,Ra,LightGBM,140,0.2583,0.2828,0.0245,0.7838
1,Lp,XGBoost,129,0.5034,0.5608,0.0574,0.9082
2,Lp,LightGBM,62,0.5327,0.5702,0.0375,0.8940


  Ra LightGBM: gap=0.0245 (pozitif)
  Lp XGBoost: gap=0.0574 (pozitif)
  Lp LightGBM: gap=0.0375 (pozitif)


## 25. Şekil Verileri (Figure Data CSVs)
Şekiller ayrı ortamda üretilir. Bu cell ham verileri CSV olarak kaydeder.


In [ ]:
# ============================================================
# 25. ŞEKİL VERİLERİ — FD1–FD4
# ============================================================

# ── FD1: Predicted vs Actual ──
fd1_rows = []
for tn in TARGETS:
    yt = stacking_results[tn]['y_test']
    yp = stacking_results[tn]['y_pred']
    for i in range(len(yt)):
        fd1_rows.append({'Target': tn, 'Actual': round(float(yt[i]),4),
                          'Predicted': round(float(yp[i]),4),
                          'Residual': round(float(yt[i]-yp[i]),4)})
pd.DataFrame(fd1_rows).to_csv(f'{OUTPUT_DIR}/FD1_Predicted_vs_Actual.csv', index=False)
print('✓ FD1 saved')

# ── FD2: Residual Distribution ──
fd2_rows = []
for tn in TARGETS:
    res = stacking_results[tn]['y_test'] - stacking_results[tn]['y_pred']
    for r in res:
        fd2_rows.append({'Target': tn, 'Residual': round(float(r),4)})
pd.DataFrame(fd2_rows).to_csv(f'{OUTPUT_DIR}/FD2_Residuals.csv', index=False)
print('✓ FD2 saved')

# ── FD3: SHAP Summary ──
fd3_rows = []
for tn in TARGETS:
    features = ra_features if tn == 'Ra' else lp_features
    sc    = stacking_results[tn]['scaler']
    model = stacking_results[tn]['model']
    Xte_s = sc.transform(df[features].values[test_idx])
    try:
        # estimators_ yapısı: [(name, fitted_estimator), ...]
        # Her tuple'ın ikinci elemanı fitted model
        rf_part = None
        for item in model.estimators_:
            est = item[1] if isinstance(item, tuple) else item
            if isinstance(est, RandomForestRegressor):
                rf_part = est
                break
        if rf_part is None:
            raise ValueError('RF not found in estimators_')
        expl = shap.TreeExplainer(rf_part)
        sv   = expl.shap_values(Xte_s)
        for fi, feat in enumerate(features):
            for obs_i in range(sv.shape[0]):
                fd3_rows.append({'Target': tn, 'Feature': feat,
                                  'Label': FEATURE_LABELS.get(feat,feat),
                                  'SHAP_value': round(float(sv[obs_i, fi]),4),
                                  'Feature_value': round(float(Xte_s[obs_i, fi]),4)})
    except Exception as e:
        print(f'SHAP hatası {tn}: {e}')
if fd3_rows:
    pd.DataFrame(fd3_rows).to_csv(f'{OUTPUT_DIR}/FD3_SHAP_Summary.csv', index=False)
    print('✓ FD3 saved')

# ── FD4: Feature Importance (RF) ──
fd4_rows = []
for tn in TARGETS:
    features = ra_features if tn == 'Ra' else lp_features
    Xf = df[features].values; y = df[tn].values
    sc = stacking_results[tn]['scaler']
    trainval_ix = np.concatenate([train_idx, val_idx])
    rf_fi = RandomForestRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
    rf_fi.fit(sc.transform(Xf[trainval_ix]), y[trainval_ix])
    for fi, feat in enumerate(features):
        fd4_rows.append({'Target': tn, 'Feature': feat,
                          'Label': FEATURE_LABELS.get(feat,feat),
                          'Importance': round(rf_fi.feature_importances_[fi],4)})
pd.DataFrame(fd4_rows).to_csv(f'{OUTPUT_DIR}/FD4_FeatureImportance.csv', index=False)
print('✓ FD4 saved')


✓ FD1 saved
✓ FD2 saved
✓ FD3 saved
✓ FD4 saved


## 26. FD5 — Partial Dependence Plot Data


In [ ]:
# ============================================================
# 26. FD5 — PARTIAL DEPENDENCE PLOT DATA
# ============================================================
from sklearn.inspection import partial_dependence

fd5_rows = []
for tn in TARGETS:
    features = ra_features if tn == 'Ra' else lp_features
    sc       = stacking_results[tn]['scaler']
    trainval_ix = np.concatenate([train_idx, val_idx])
    rf_pdp = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
    Xf     = df[features].values
    y      = df[tn].values
    rf_pdp.fit(sc.transform(Xf[trainval_ix]), y[trainval_ix])
    for fi in range(len(features)):
        pd_res = partial_dependence(rf_pdp, sc.transform(Xf[trainval_ix]),
                                     features=[fi], grid_resolution=50)
        for grid_val, avg_pred in zip(pd_res['grid_values'][0], pd_res['average'][0]):
            fd5_rows.append({'Target': tn, 'Feature': features[fi],
                              'Label': FEATURE_LABELS.get(features[fi], features[fi]),
                              'Grid_value': round(float(grid_val),4),
                              'Avg_prediction': round(float(avg_pred),4)})

pd.DataFrame(fd5_rows).to_csv(f'{OUTPUT_DIR}/FD5_PDP.csv', index=False)
print('✓ FD5 saved')


✓ FD5 saved


## 27. FD6 — Learning Curve (3 CV Stratejisi)


In [ ]:
# ============================================================
# 27. FD6 — LEARNING CURVE
# Model: RF | 3 CV stratejisi karşılaştırma
# ============================================================
TRAIN_FRACS = np.linspace(0.1, 1.0, 10)
fd6_rows    = []

for tn in TARGETS:
    features = ra_features if tn == 'Ra' else lp_features
    X_all    = df[features].values
    y_all    = df[tn].values
    X_tr_all = X_all[train_idx]
    y_tr_all = y_all[train_idx]

    for frac in TRAIN_FRACS:
        n_use = max(10, int(frac * len(train_idx)))
        idx_use = train_idx[:n_use]
        X_use = X_all[idx_use]; y_use = y_all[idx_use]
        g_use = df['Block'].values[idx_use]

        rf_lc = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)

        # CV1
        kf_sc = cv_kfold(lambda: rf_lc, X_use, y_use, k=min(5, len(np.unique(g_use))))
        fd6_rows.append({'Target': tn, 'CV': 'CV1_KFold', 'frac': round(frac,2),
                          'n_train': n_use, 'R2': round(kf_sc[0]['R2'],4), 'RMSE': round(kf_sc[0]['RMSE'],4)})

        # CV2 — min 2 unique groups gerekli
        n_uniq = len(np.unique(g_use))
        if n_uniq >= 2:
            gkf_sc = cv_group_kfold(lambda: rf_lc, X_use, y_use, g_use, k=min(n_uniq,16))
            fd6_rows.append({'Target': tn, 'CV': 'CV2_GKF', 'frac': round(frac,2),
                              'n_train': n_use, 'R2': round(gkf_sc[0]['R2'],4), 'RMSE': round(gkf_sc[0]['RMSE'],4)})

        # CV3
        ch_sc = cv_chrono(lambda: rf_lc, X_use, y_use, g_use)
        fd6_rows.append({'Target': tn, 'CV': 'CV3_Chrono', 'frac': round(frac,2),
                          'n_train': n_use, 'R2': round(ch_sc[0]['R2'],4), 'RMSE': round(ch_sc[0]['RMSE'],4)})

pd.DataFrame(fd6_rows).to_csv(f'{OUTPUT_DIR}/FD6_LearningCurve.csv', index=False)
print('✓ FD6 saved')


✓ FD6 saved


## 28. FD7 — SHAP Interaction Values


In [ ]:
# ============================================================
# 28. FD7 — SHAP INTERACTION VALUES
# ============================================================
fd7_rows = []

for tn in TARGETS:
    features  = ra_features if tn == 'Ra' else lp_features
    sc        = stacking_results[tn]['scaler']
    Xte_s     = sc.transform(df[features].values[test_idx])
    trainval_ix = np.concatenate([train_idx, val_idx])
    Xtr_s     = sc.transform(df[features].values[trainval_ix])
    y_t       = df[tn].values[trainval_ix]

    rf_shap = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
    rf_shap.fit(Xtr_s, y_t)
    expl = shap.TreeExplainer(rf_shap)
    sv_i = expl.shap_interaction_values(Xte_s)

    for fi in range(len(features)):
        for fj in range(len(features)):
            mean_inter = float(np.abs(sv_i[:, fi, fj]).mean())
            fd7_rows.append({'Target': tn,
                              'Feature_i': features[fi], 'Label_i': FEATURE_LABELS.get(features[fi],features[fi]),
                              'Feature_j': features[fj], 'Label_j': FEATURE_LABELS.get(features[fj],features[fj]),
                              'Mean_Abs_Interaction': round(mean_inter,4)})

pd.DataFrame(fd7_rows).to_csv(f'{OUTPUT_DIR}/FD7_SHAP_Interaction.csv', index=False)
print('✓ FD7 saved')


## 29. Model Kaydet + Kalite Kontrol


In [ ]:
# ============================================================
# 29. MODEL KAYDET + KALİTE KONTROL
# ============================================================
import zipfile, glob

for tn in TARGETS:
    with open(f'{OUTPUT_DIR}/stacking_model_{tn}.pkl','wb') as f:
        pickle.dump(stacking_results[tn]['model'], f)
    with open(f'{OUTPUT_DIR}/scaler_{tn}.pkl','wb') as f:
        pickle.dump(stacking_results[tn]['scaler'], f)

expected_files = [
    'Table1_Descriptive.csv', 'Table2_PhysicsFeatures.csv',
    'Table3_VIF.csv', 'Table3b_Correlation.csv',
    'Table4a_BorutaPerc.csv', 'Table4_FeatureSelection.csv',
    'Table5_CV_Ra.csv', 'Table5_CV_Lp.csv',
    'Table6_BaseModel_Ra.csv', 'Table6_BaseModel_Lp.csv',
    'Table7_BO_Ra.csv', 'Table7_BO_Lp.csv',
    'Table8_Bootstrap_CI.csv',
    'Table9_Wilcoxon.csv',
    'Table10_Permutation.csv',
    'Table11_MetaLearner.csv',
    'Table12_EarlyStopping.csv',
    'FD1_Predicted_vs_Actual.csv', 'FD2_Residuals.csv',
    'FD3_SHAP_Summary.csv', 'FD4_FeatureImportance.csv',
    'FD5_PDP.csv', 'FD6_LearningCurve.csv', 'FD7_SHAP_Interaction.csv',
    'stacking_model_Ra.pkl', 'stacking_model_Lp.pkl',
    'scaler_Ra.pkl', 'scaler_Lp.pkl',
]

print('KALİTE KONTROL:')
all_ok = True
for fname in expected_files:
    fpath = f'{OUTPUT_DIR}/{fname}'
    ok = os.path.exists(fpath)
    if not ok: all_ok = False
    print(f'  {"✓" if ok else "✗ EKSİK"} {fname}')

# ZIP
zip_path = f'{OUTPUT_DIR}/pap_AG_VBD_milling_ML_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in expected_files:
        fpath = f'{OUTPUT_DIR}/{fname}'
        if os.path.exists(fpath):
            zf.write(fpath, fname)

print(f'\n{"✓ TÜM DOSYALAR MEVCUT" if all_ok else "EKSİK DOSYALAR VAR — kontrol et"}')
print(f'✓ ZIP: {zip_path}')

# Final metrikler
print('\n── FINAL STACKING METRİKLERİ ──')
for tn in TARGETS:
    m = stacking_results[tn]['metrics']
    print(f'  {tn}: R²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  MAPE={m["MAPE"]:.2f}%')
print('\n✓ pap_AG_VBD_milling_ML pipeline tamamlandı!')
